# Draft strategy

What roster construction is actually worth in **these two leagues**, as live queries. Re-run the
notebook and the numbers update — nothing below is transcribed by hand.

The question that started this: a "full house" opening — **three running backs and two receivers in
the first five rounds** — on the theory that running back is the scarce position and that even elite
receivers can't carry a team the way an elite back can.

Half of that premise turns out to be true and the conclusion drawn from it doesn't follow. The
scarcity is real, but it is *shallow*: it lives in the first two rounds and is gone by the third,
and a strategy that spends three of five premium picks chasing it is buying the part of the running
back curve that has already flattened. Sections 2-4 measure the premise; section 5 settles the
strategy by simulating ~81,000 drafts off the real historical ADP boards; sections 6-8 are the
honesty checks on that result, one of which changes the recommendation.

**Contents**
1. [The two leagues — and the superflex question](#leagues)
2. [Is the premise true? Positional scarcity](#scarcity)
3. [What each round actually returned](#rounds)
4. [Where mid-round running backs go wrong](#hitrate)
5. [The test: simulate the draft](#simulation)
6. [Is any of it significant?](#significance)
7. [Ordering: RB early vs RB often](#ordering)
8. [Who else is at the table](#field)
9. [The superflex counterfactual](#superflex)
10. [What to actually do](#doing)

In [1]:
# Find the repo root from wherever the kernel started, so `src` imports work.
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
from scipy import stats

from src.query import q, tables, columns, peek

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)

tables("gold").query("table.str.startswith('draft_strategy')")

,table,layer,rows
6,draft_strategy_results,gold,81312
7,draft_strategy_summary,gold,376


<a id="leagues"></a>
## 1. The two leagues — and the superflex question

**Neither league is superflex.** That is worth stating first, because it is the premise of the "do I
need to take quarterbacks earlier?" worry and it does not hold. Both rows below come from
`league_settings`, which reads each platform's own settings rather than anything typed in by hand:
Sleeper's `roster_positions` contains no `SUPER_FLEX` entry, and ESPN's lineup slot 7 (`OP`, its
superflex slot) is set to zero.

So both leagues start exactly one quarterback. Section 9 runs the counterfactual anyway — if either
league *did* add a superflex slot the answer would change completely, and by a lot — but under the
settings these leagues actually have, quarterback is a one-slot position.

The two leagues differ in ways that matter more than the format label suggests, and they push in
opposite directions:

- **Sleeper** is 12 teams, half-PPR, with **two** flex slots — 8 skill starters per team, so 96
  skill players start every week.
- **ESPN** is 10 teams, full PPR, with **one** flex — 7 skill starters, so only 70 start.

More teams and more flex slots make the pool thinner; full PPR lifts receivers. That is why the same
strategy does not win both leagues.

In [2]:
q("""
    SELECT
        league_key                          AS league,
        team_count                          AS teams,
        rec_pts                             AS "pts/reception",
        qb_slots || ' QB'                   AS qb,
        rb_slots || ' RB'                   AS rb,
        wr_slots || ' WR'                   AS wr,
        te_slots || ' TE'                   AS te,
        flex_slots || ' FLEX'               AS flex,
        superflex_slots                     AS "superflex slots",
        bench_slots                         AS bench,
        qb_slots + rb_slots + wr_slots + te_slots + flex_slots + superflex_slots
                                            AS "skill starters",
        team_count * (qb_slots + rb_slots + wr_slots + te_slots + flex_slots + superflex_slots)
                                            AS "skill players starting"
    FROM league_settings
    ORDER BY league_key
""").set_index("league").T

league,espn,sleeper
teams,10,12
pts/reception,1.0,0.5
qb,1 QB,1 QB
rb,2 RB,2 RB
wr,2 WR,2 WR
te,1 TE,1 TE
flex,1 FLEX,2 FLEX
superflex slots,0,0
bench,7,5
skill starters,7,8


In [3]:
# The superflex claim, checked rather than asserted.
sf = q("SELECT league_key, superflex_slots, qb_slots FROM league_settings ORDER BY league_key")
for _, row in sf.iterrows():
    verdict = (
        f"{row.superflex_slots} superflex slot(s) — QB is flex-eligible"
        if row.superflex_slots
        else f"no superflex slot: {row.qb_slots} starting QB, and QB cannot fill a flex spot"
    )
    print(f"{row.league_key:>8}: {verdict}")

print(
    "\nsuperflex leagues among mine:",
    int((sf.superflex_slots > 0).sum()), "of", len(sf),
)

    espn: no superflex slot: 1 starting QB, and QB cannot fill a flex spot
 sleeper: no superflex slot: 1 starting QB, and QB cannot fill a flex spot

superflex leagues among mine: 0 of 2


<a id="scarcity"></a>
## 2. Is the premise true? Positional scarcity

**Partly.** An elite running back really is worth more than an elite receiver — and the gap closes
fast.

The table below is points over replacement by *realised* positional finish, averaged over 2015-2025,
in each league's own scoring. `RB-WR` is the whole argument in one column: how much more the RB
finishing Nth was worth than the WR finishing Nth.

Read where that column crosses zero. Above the crossover, running back is the scarcer asset and the
"full house" premise holds. Below it, the premise inverts and receivers are worth *more* at the same
rank — because so many more of them start (`starters_at_position` in `points_over_replacement`
counts dedicated slots plus the flex spots that position actually won), which pushes the receiver
replacement level far deeper down the list.

The crossover lands in a very different place in each league, and that difference is most of why
these two leagues want different drafts.

In [4]:
def por_curve(league_key, max_rank=30):
    return q("""
        SELECT position_rank AS rank,
               MAX(CASE WHEN position = 'RB' THEN por END) AS RB,
               MAX(CASE WHEN position = 'WR' THEN por END) AS WR,
               MAX(CASE WHEN position = 'TE' THEN por END) AS TE,
               MAX(CASE WHEN position = 'QB' THEN por END) AS QB
        FROM (
            SELECT position, position_rank, AVG(points_over_replacement) AS por
            FROM points_over_replacement
            WHERE league_key = ? AND season <= (SELECT MAX(season) FROM weekly_stats)
            GROUP BY position, position_rank
        )
        GROUP BY position_rank
        HAVING position_rank <= ?
        ORDER BY position_rank
    """, [league_key, max_rank])


curves = {}
for league_key in ("sleeper", "espn"):
    curve = por_curve(league_key).set_index("rank")
    curve["RB-WR"] = curve.RB - curve.WR
    curves[league_key] = curve

pd.concat(curves, axis=1).round(1)

sleeper                              espn                           
          RB     WR     TE     QB RB-WR     RB     WR     TE     QB RB-WR
rank                                                                     
1      218.2  176.7  110.9  134.1  41.6  205.3  181.8  121.7  114.3  23.5
2      176.3  149.7   79.1   95.5  26.5  153.1  145.4   86.9   76.0   7.7
3      159.9  132.4   62.3   86.0  27.4  137.4  130.8   63.4   67.0   6.5
4      146.3  118.6   53.3   75.1  27.7  119.2  114.4   51.2   54.3   4.9
5      128.6  111.7   43.6   60.1  16.8  102.6  103.4   40.2   39.0  -0.8
6      118.6  102.5   35.3   45.9  16.1   92.5   94.5   31.0   26.7  -2.0
7      105.8   92.1   27.3   39.5  13.7   75.1   82.8   18.4   20.7  -7.8
8       95.5   86.1   22.1   34.9   9.4   66.0   75.3   14.6   15.4  -9.3
9       87.3   82.1   18.7   29.3   5.2   58.9   69.2   10.6   11.0 -10.3
10      83.4   77.7   12.8   25.2   5.7   53.5   63.0    3.3    5.5  -9.5
11      78.6   73.1    6.5   19.9   5.5   47.0   60.3    0.0    0.0 -13.3
12      72.4   70.8    3.4   10.3   1.6   41.9   56.4   -5.3   -9.9 -14.5
13      69.2   68.3    1.1    0.0   0.9   36.3   52.2  -10.4  -22.4 -15.8
14      66.2   66.1   -3.0   -6.1   0.1   32.2   49.4  -16.7  -25.5 -17.2
15      60.4   60.1   -6.8  -13.6   0.3   27.5   46.2  -20.0  -32.8 -18.7
16      55.9   56.3   -9.8  -19.3  -0.4   24.6   40.1  -24.3  -37.5 -15.5
17      52.3   53.4  -12.8  -25.5  -1.1   20.6   36.4  -27.0  -45.5 -15.8
18      49.1   51.8  -14.5  -33.5  -2.7   15.5   32.9  -30.3  -53.9 -17.4
19      46.2   49.6  -17.2  -36.8  -3.5   12.3   30.5  -34.1  -58.8 -18.2
20      42.3   46.4  -20.4  -44.6  -4.1    8.9   28.0  -36.5  -64.4 -19.1
21      39.3   45.0  -24.6  -54.5  -5.7    5.5   26.6  -40.6  -73.8 -21.0
22      36.5   43.5  -26.5  -64.1  -7.0    0.5   22.9  -42.9  -83.9 -22.3
23      32.5   41.2  -28.5  -72.4  -8.7   -3.0   19.2  -47.2  -92.2 -22.2
24      27.7   37.8  -31.8  -78.5 -10.1   -7.8   17.4  -50.2  -97.4 -25.1
25      23.2   35.1  -34.1  -88.2 -11.9  -11.0   13.4  -54.4 -107.7 -24.5
26      19.3   32.3  -37.4  -95.3 -12.9  -14.6   10.8  -58.7 -112.7 -25.4
27      17.0   29.7  -40.3 -102.6 -12.6  -17.5    7.4  -62.3 -122.3 -24.9
28      12.8   27.9  -42.9 -113.5 -15.1  -20.9    4.8  -64.4 -132.2 -25.6
29       9.0   25.5  -44.9 -125.8 -16.5  -26.1    2.2  -65.7 -143.5 -28.3
30       7.4   24.7  -47.0 -138.7 -17.3  -28.8    0.2  -68.1 -157.0 -29.0

In [5]:
# Where does the running back premium run out? First rank at which the WR finishing there is worth
# more than the RB finishing there.
for league_key, curve in curves.items():
    ahead = curve.index[curve["RB-WR"] > 0]
    crossover = int(ahead.max()) + 1 if len(ahead) else 1
    top2 = curve.loc[:2, "RB-WR"].mean()
    print(
        f"{league_key:>8}: RB is worth more than WR through rank {crossover - 1}, "
        f"WR from rank {crossover} on. "
        f"Premium at the top (ranks 1-2): {top2:+.0f} pts."
    )

 sleeper: RB is worth more than WR through rank 15, WR from rank 16 on. Premium at the top (ranks 1-2): +34 pts.
    espn: RB is worth more than WR through rank 4, WR from rank 5 on. Premium at the top (ranks 1-2): +16 pts.


<a id="rounds"></a>
## 3. What each round actually returned

Section 2 measured finishes. This one measures **picks**, which is the thing a strategy actually
controls: for every player with a preseason consensus ADP since 2015, what did a pick in that round
go on to return? Value is `draft_value.actual_value` — points over replacement in that league's
scoring, floored at zero, because nobody is forced to start a player worse than the waiver wire — and
a drafted player who never played a snap is carried as a genuine zero rather than dropped, so the
bust rate is priced in.

Rounds are that league's own team count, so "round 3" means picks 25-36 in Sleeper and 21-30 in ESPN.

The pattern is the same one section 2 predicted, now in draft-pick terms. Running back wins the
early rounds in Sleeper and gives the lead up from round 3. In full-PPR ESPN it barely leads at all,
and from round 3 on receivers are clearly the better buy.

Rounds 3-5 are exactly where a "full house" opening spends its third running back.

In [6]:
def value_by_round(league_key, rounds=8):
    return q("""
        SELECT CAST(CEIL(d.consensus_adp / l.team_count) AS INT) AS round,
               AVG(CASE WHEN position = 'RB' THEN actual_value END) AS RB,
               AVG(CASE WHEN position = 'WR' THEN actual_value END) AS WR,
               AVG(CASE WHEN position = 'TE' THEN actual_value END) AS TE,
               AVG(CASE WHEN position = 'QB' THEN actual_value END) AS QB,
               COUNT(*) AS picks
        FROM draft_value d
        JOIN league_settings l ON l.league_key = d.league_key
        WHERE d.league_key = ? AND d.actual_value IS NOT NULL
          AND d.consensus_adp <= l.team_count * ?
        GROUP BY 1 ORDER BY 1
    """, [league_key, rounds])


returns = {k: value_by_round(k).set_index("round") for k in ("sleeper", "espn")}
pd.concat(returns, axis=1).round(1)

sleeper                          espn                         
           RB    WR    TE    QB picks    RB    WR     TE    QB picks
round                                                               
1        98.6  92.9  83.6   NaN   130  79.6  80.8   83.9   NaN    98
2        75.3  63.0  61.2  65.4   130  61.8  71.5  105.6  19.0   119
3        48.5  52.4  51.2  57.6   131  33.8  35.0   37.5  61.9    99
4        32.3  44.2  50.9  29.1   132  25.8  41.9   55.4  40.4   119
5        34.1  38.5  19.3  36.4   129  14.6  37.9   36.2  19.0   115
6        24.3  25.5  26.8  38.7   129  16.4  23.0   19.2  27.0   102
7        23.2  21.3   9.4  19.1   129  11.8  20.8   24.6  29.0   112
8         8.9  17.2   9.9  29.0   123   8.4  12.3    7.5   6.4   103

In [7]:
# Which position returned more, per round, per league.
for league_key, table in returns.items():
    rb_rounds = table.index[table.RB > table.WR].tolist()
    print(f"{league_key:>8}: RB out-returned WR in round(s) {rb_rounds or 'none'} "
          f"of the first {len(table)}")

 sleeper: RB out-returned WR in round(s) [1, 2, 7] of the first 8
    espn: RB out-returned WR in round(s) none of the first 8


<a id="hitrate"></a>
## 4. Where mid-round running backs go wrong

The averages above hide *how* they happen, and the how is the useful part. `boom_bust` classifies
every drafted player-season by whether he finished as a top-`team_count` asset at his position — an
absolute measure, not "did he beat his ADP" — so this is the hit rate on a pick.

Rounds 1-3 look similar for both positions. The separation is in **rounds 4-5**: a running back
taken there hits far less often than a receiver taken there, in both leagues. That is the specific
failure mode of the full-house opening. It isn't that the third running back is a bad player, it's
that the fourth- and fifth-round running back is the least reliable pick on the board, and the
strategy commits to taking one before the draft starts.

`n` columns are shown because the tight end row is thin — few tight ends go early enough to appear —
and shouldn't be read as a finding.

In [8]:
def hit_rate(league_key, rounds=6):
    return q("""
        SELECT CAST(CEIL(b.consensus_adp / l.team_count) AS INT) AS round,
               100 * AVG(CASE WHEN position = 'RB' THEN is_elite_finish::INT END) AS "RB hit%",
               100 * AVG(CASE WHEN position = 'WR' THEN is_elite_finish::INT END) AS "WR hit%",
               SUM(CASE WHEN position = 'RB' THEN 1 ELSE 0 END) AS "n RB",
               SUM(CASE WHEN position = 'WR' THEN 1 ELSE 0 END) AS "n WR",
               100 * AVG(CASE WHEN position = 'TE' THEN is_elite_finish::INT END) AS "TE hit%",
               100 * AVG(CASE WHEN position = 'QB' THEN is_elite_finish::INT END) AS "QB hit%"
        FROM boom_bust b
        JOIN league_settings l ON l.league_key = b.league_key
        WHERE b.league_key = ? AND b.consensus_adp <= l.team_count * ?
        GROUP BY 1 ORDER BY 1
    """, [league_key, rounds])


hits = {k: hit_rate(k).set_index("round") for k in ("sleeper", "espn")}
pd.concat(hits, axis=1).round(1)

sleeper                                        espn                                    
      RB hit% WR hit%  n RB  n WR TE hit% QB hit% RB hit% WR hit%  n RB  n WR TE hit% QB hit%
round                                                                                        
1        61.0    63.3  77.0  49.0   100.0     NaN    50.8    55.9  61.0  34.0   100.0     NaN
2        50.9    47.3  53.0  55.0    62.5    76.9    48.1    51.9  54.0  52.0    83.3    66.7
3        32.6    31.0  46.0  58.0    76.9    76.9    35.3    17.8  34.0  45.0    50.0    80.0
4        13.3    25.4  45.0  59.0    90.9    57.1    25.6    28.0  43.0  50.0    70.0    66.7
5        17.1    21.2  41.0  52.0    35.3    55.6     8.6    27.3  35.0  55.0    72.7    42.9
6        12.5    11.8  40.0  51.0    53.3    61.9    11.4    10.8  35.0  37.0    38.5    56.2

In [9]:
for league_key, table in hits.items():
    late = table.loc[4:5]
    print(f"{league_key:>8}: rounds 4-5 hit rate — RB {late['RB hit%'].mean():.1f}% "
          f"vs WR {late['WR hit%'].mean():.1f}%")

 sleeper: rounds 4-5 hit rate — RB 15.2% vs WR 23.3%
    espn: rounds 4-5 hit rate — RB 17.1% vs WR 27.6%


<a id="simulation"></a>
## 5. The test: simulate the draft

Everything above is circumstantial. A draft strategy is a claim about *roster construction*, and the
only way to settle it is to run the draft — so `src/gold/draft_strategy.py` does, ~81,000 times.

Each simulated draft is a snake draft off that season's real `adp_consensus` board, with every team
scored on the hindsight-best starting lineup it could field from the roster it ended up with, using
that league's own scoring. One team — the focal team — follows a strategy for the first five rounds;
everyone else takes the best player left by ADP. Then the focal seat rotates through every draft
slot, and the whole thing repeats for every season from 2015 on.

A **strategy** here is a *composition*: how many of each position in the first five rounds, with ADP
deciding the order. That is what a human actually does — nobody committed to "3 RB, 2 WR" passes the
top receiver on the board at 1.03 to force a back — and it keeps the strategy space at 36 instead of
4⁵ = 1024. `ADP` is the control: no constraint at all, best available for all fifteen-ish rounds.

`points_vs_field` is the focal team's starting-lineup total minus the average of the other teams in
that same draft, so the control sits at exactly zero by construction. Full details and caveats are
in the module docstring.

In [10]:
q("""
    SELECT league_key AS league, variant, field_model AS field,
           COUNT(*) AS drafts,
           COUNT(DISTINCT strategy) AS strategies,
           COUNT(DISTINCT season) AS seasons,
           MIN(season) AS "from", MAX(season) AS "to"
    FROM draft_strategy_results
    GROUP BY ALL ORDER BY league, variant, field
""")

,league,variant,field,drafts,strategies,seasons,from,to
0,espn,actual,adp,6270,57,11,2015,2025
1,espn,actual,mixed,12210,37,11,2015,2025
2,espn,superflex,adp,6270,57,11,2015,2025
3,espn,superflex,mixed,12210,37,11,2015,2025
4,sleeper,actual,adp,7524,57,11,2015,2025
5,sleeper,actual,mixed,14652,37,11,2015,2025
6,sleeper,superflex,adp,7524,57,11,2015,2025
7,sleeper,superflex,mixed,14652,37,11,2015,2025


In [11]:
def ranking(league_key, variant="actual", field_model="adp"):
    table = q("""
        SELECT strategy, points_vs_field, finish_rank, 100 * win_rate AS "win%",
               100 * top_third_rate AS "top3rd%", t_stat, seasons_positive, n_seasons
        FROM draft_strategy_summary
        WHERE league_key = ? AND variant = ? AND field_model = ?
          AND strategy_kind <> 'ordering'
        ORDER BY points_vs_field DESC
    """, [league_key, variant, field_model]).reset_index(drop=True)
    table.index = range(1, len(table) + 1)
    return table


for league_key in ("sleeper", "espn"):
    table = ranking(league_key)
    keep = pd.concat([
        table.head(5),
        table[table.strategy.isin(["ADP", "3RB2WR", "2RB3WR"])],
        table.tail(3),
    ]).drop_duplicates("strategy").sort_values("points_vs_field", ascending=False)
    print(f"\n=== {league_key} — opening composition, best to worst (of {len(table)}) ===")
    print(keep.round(2).to_string())


=== sleeper — opening composition, best to worst (of 37) ===
     strategy  points_vs_field  finish_rank   win%  top3rd%  t_stat  seasons_positive  n_seasons
1      2RB3WR            27.85         5.86  12.88    38.64    2.58                 8         11
2   1RB3WR1TE            24.36         6.13   9.85    38.64    1.65                 7         11
3      1RB4WR            23.70         5.96  12.12    37.88    1.55                 8         11
4   2RB2WR1TE            19.40         6.14   9.85    34.85    2.19                 8         11
5   1QB1RB3WR            12.10         6.27   9.85    37.88    0.95                 6         11
10        ADP             0.00         6.50   8.33    33.33    0.17                 4         11
12     3RB2WR            -2.61         6.58   7.58    33.33   -0.28                 5         11
35  2QB2WR1TE           -57.72         7.54   3.03    22.73   -2.57                 2         11
36  1QB2WR2TE           -58.08         7.65   6.82    22.73   -2.


=== espn — opening composition, best to worst (of 37) ===
        strategy  points_vs_field  finish_rank   win%  top3rd%  t_stat  seasons_positive  n_seasons
1   1QB1RB2WR1TE            41.45         5.11  11.82    47.27    2.10                 7         11
2   1QB1RB1WR2TE            31.33         5.13  13.64    45.45    1.39                 5         11
3      1RB2WR2TE            27.48         5.06  11.82    46.36    1.49                 7         11
4   1QB2RB1WR1TE            22.84         5.25  11.82    45.45    1.16                 6         11
5      2RB2WR1TE            22.41         5.20  13.64    42.73    2.14                 8         11
16           ADP             0.00         5.50  10.00    40.00    0.94                 6         11
22        2RB3WR            -6.05         5.79   9.09    34.55   -0.96                 4         11
27        3RB2WR           -16.67         5.81  10.00    34.55   -1.99                 3         11
35        1QB4WR           -40.59        

In [12]:
# Where the full house actually lands, and what the control would have paid instead.
for league_key in ("sleeper", "espn"):
    table = ranking(league_key)
    place = {s: int(table.index[table.strategy == s][0]) for s in ("3RB2WR", "ADP")}
    full_house = table.loc[place["3RB2WR"]]
    print(
        f"{league_key:>8}: 3RB2WR ranks {place['3RB2WR']}/{len(table)} "
        f"({full_house.points_vs_field:+.1f} pts vs field, "
        f"{full_house.seasons_positive}/{full_house.n_seasons} seasons positive) — "
        f"the do-nothing ADP control ranks {place['ADP']}."
    )

 sleeper: 3RB2WR ranks 12/37 (-2.6 pts vs field, 5/11 seasons positive) — the do-nothing ADP control ranks 10.


    espn: 3RB2WR ranks 27/37 (-16.7 pts vs field, 3/11 seasons positive) — the do-nothing ADP control ranks 16.


<a id="significance"></a>
## 6. Is any of it significant?

This is the section that keeps the one above honest, and it pulls the conclusion back a long way.

Thirty-seven strategies were ranked. At the usual threshold roughly two of them would clear |t| > 2
by chance alone, so picking the top row and calling it the best strategy is exactly the mistake that
sampling noise is designed to produce. And the t-tests below are already the *conservative* version:
they run on the eleven per-season means rather than the ~1,500 individual drafts, because drafts
within one season share the same player outcomes — one running back tearing an ACL moves every
RB-heavy draft that year together — and treating those as independent would shrink the standard
error by roughly a factor of the number of draft slots and make almost everything "significant".

So instead of trusting any single strategy, the test below asks a **trend** question, which has far
more power: holding everything else loose, does adding one more of a position to the opening help or
hurt? Each cell is the mean `points_vs_field` across every strategy with that many of that position;
the slope is fitted per season and t-tested across seasons.

Under the leagues' **actual** settings, **no position's slope clears significance.** Not running
back, not receiver, not quarterback; the nearest miss is tight end in ESPN, and it misses. What the
cells do show — consistently, in both leagues — is an inverted U: zero of a position is bad, four or
five of it is worse, and the middle is flat. The honest reading is that opening composition is worth
roughly one good waiver claim, and that the only genuinely costly openings are the extreme ones.

The cells below make that concrete. Somewhat more strategies clear |t| > 2 than chance alone would
produce, so there is *some* signal in the sweep — but it is thin, and most of it sits on the negative
side: what the simulation identifies reliably is which openings lose. Under the actual settings the
losers are the all-in ones (five backs, five receivers) in ESPN and the quarterback/tight-end-heavy
ones in Sleeper.

On the positive side only two openings clear the bar in either league, and **exactly one clears it in
both** — `2RB2WR1TE`. Two backs, two receivers and a tight end: a balanced opening, arrived at from
two different scoring formats and two different team counts. That is the closest thing to a robust
positive finding in this notebook, and it is worth more than the top row of section 5, which is
league-specific and one lucky season away from being a different row.

That is a real answer to the original question, just not an exciting one: **the full house isn't
wrong so much as it isn't worth committing to**, and the thing it gets wrong — three backs — is on
the losing side of a slope too shallow to prove.

In [13]:
compositions = q("""
    SELECT * FROM draft_strategy_results
    WHERE strategy_kind = 'composition' AND field_model = 'adp'
""")
for position in ("QB", "RB", "WR", "TE"):
    compositions[position] = (
        compositions.strategy.str.extract(rf"(\d){position}").fillna(0).astype(int)
    )


def trend(frame, position):
    """Mean vs-field by count, plus a season-clustered test of the slope."""
    per_season = frame.groupby(["season", position]).points_vs_field.mean().reset_index()
    slopes = [
        np.polyfit(group[position], group.points_vs_field, 1)[0]
        for _, group in per_season.groupby("season")
        if group[position].nunique() > 1
    ]
    t_stat, p_value = stats.ttest_1samp(slopes, 0.0)
    cells = frame.groupby(position).points_vs_field.mean()
    return {
        "position": position,
        **{f"{n} in opening": cells.get(n, np.nan) for n in range(6)},
        "pts/extra pick": np.mean(slopes),
        "t": t_stat,
        "p": p_value,
    }


for league_key in ("sleeper", "espn"):
    frame = compositions[
        (compositions.league_key == league_key) & (compositions.variant == "actual")
    ]
    print(f"\n=== {league_key} / actual — value of the Nth pick spent on a position ===")
    print(pd.DataFrame([trend(frame, p) for p in ("QB", "RB", "WR", "TE")])
          .set_index("position").round(2).to_string())


=== sleeper / actual — value of the Nth pick spent on a position ===
          0 in opening  1 in opening  2 in opening  3 in opening  4 in opening  5 in opening  pts/extra pick     t     p
position                                                                                                                
QB               -5.20        -13.26        -31.56           NaN           NaN           NaN          -13.18 -1.72  0.12
RB              -44.16         -3.89          1.53         -5.87        -15.74        -18.63            2.42  0.26  0.80
WR              -17.93        -16.36        -11.87         -8.41        -13.45        -26.88           -0.93 -0.11  0.91
TE               -6.94        -11.48        -31.04           NaN           NaN           NaN          -12.05 -1.72  0.12

=== espn / actual — value of the Nth pick spent on a position ===
          0 in opening  1 in opening  2 in opening  3 in opening  4 in opening  5 in opening  pts/extra pick     t     p
position        

In [14]:
# Which strategies clear |t| > 2, and which way do they point?
significant = q("""
    SELECT league_key AS league, variant,
           COUNT(*) AS strategies,
           ROUND(0.05 * COUNT(*), 1) AS "expected by chance",
           SUM((ABS(t_stat) > 2)::INT) AS "|t|>2",
           SUM((t_stat > 2)::INT) AS good,
           SUM((t_stat < -2)::INT) AS bad
    FROM draft_strategy_summary
    WHERE field_model = 'adp' AND strategy_kind <> 'ordering'
    GROUP BY ALL ORDER BY league, variant
""")
print(significant.to_string(index=False))

named = q("""
    SELECT league_key, variant, t_stat > 2 AS good, strategy
    FROM draft_strategy_summary
    WHERE field_model = 'adp' AND strategy_kind <> 'ordering' AND ABS(t_stat) > 2
    ORDER BY league_key, variant, t_stat DESC
""")
for (league_key, variant), group in named.groupby(["league_key", "variant"]):
    for good in (True, False):
        names = group[group.good == good].strategy.tolist()
        if names:
            print(f"\n{league_key:>8} / {variant:<9} significantly "
                  f"{'GOOD' if good else 'BAD ':<4}: {', '.join(names)}")

 league   variant  strategies  expected by chance  |t|>2  good  bad
   espn    actual          37                 1.9    5.0   2.0  3.0
   espn superflex          37                 1.9   11.0   7.0  4.0
sleeper    actual          37                 1.9    6.0   2.0  4.0
sleeper superflex          37                 1.9   14.0  10.0  4.0



    espn / actual    significantly GOOD: 2RB2WR1TE, 1QB1RB2WR1TE

    espn / actual    significantly BAD : 5WR, 4RB1WR, 5RB

    espn / superflex significantly GOOD: 2QB1RB1WR1TE, 2QB1RB2WR, 1QB2RB2WR, 2QB2RB1WR, 1QB1RB2WR1TE, 2QB2RB1TE, 1QB1RB3WR

    espn / superflex significantly BAD : 4RB1TE, 4RB1WR, 5RB, 3RB2WR

 sleeper / actual    significantly GOOD: 2RB3WR, 2RB2WR1TE

 sleeper / actual    significantly BAD : 1QB2WR2TE, 2QB2WR1TE, 2QB1RB2TE, 2QB1WR2TE

 sleeper / superflex significantly GOOD: 2QB2RB1TE, 2QB1RB2WR, 2QB2RB1WR, 2QB1RB1WR1TE, 1QB2RB2WR, 2QB3RB, 1QB1RB3WR, 2QB1RB2TE, 1QB1RB2WR1TE, 1QB2RB1WR1TE

 sleeper / superflex significantly BAD : 1RB2WR2TE, 3RB2TE, 3WR2TE, 2RB1WR2TE


In [15]:
# The only claim that survives being asked of both leagues at once.
actual = named[(named.variant == "actual") & named.good]
in_both = set.intersection(*(
    set(group.strategy) for _, group in actual.groupby("league_key")
))
print("openings significantly better than the field in BOTH leagues:", ", ".join(sorted(in_both)) or "none")

q("""
    SELECT league_key AS league, strategy, points_vs_field, finish_rank,
           100 * top_third_rate AS "top3rd%", t_stat, seasons_positive, n_seasons
    FROM draft_strategy_summary
    WHERE variant = 'actual' AND field_model = 'adp' AND strategy = ?
    ORDER BY league_key
""", [sorted(in_both)[0]]).round(2) if in_both else None

openings significantly better than the field in BOTH leagues: 2RB2WR1TE


,league,strategy,points_vs_field,finish_rank,top3rd%,t_stat,seasons_positive,n_seasons
0,espn,2RB2WR1TE,22.41,5.20,42.73,2.14,8,11
1,sleeper,2RB2WR1TE,19.40,6.14,34.85,2.19,8,11


<a id="ordering"></a>
## 7. Ordering: RB early vs RB often

If composition barely matters, does the *order* within it? This is the part of the full-house pitch
that survives.

Holding the composition fixed and forcing an exact position sequence — every distinct permutation of
3RB+2WR and of 2RB+3WR — separates rosters that section 5 could not tell apart at all. And one
pattern holds without exception in both leagues: **every ordering that opens with a running back
beats every ordering that opens with two receivers.** The third cell checks that claim rather than
trusting the eye, since "without exception" is exactly the kind of statement that quietly stops being
true after a rebuild.

That is the defensible version of the original idea. The scarcity measured in section 2 is real and
it is concentrated in the first two rounds, so the way to capture it is to take a back *early* — not
to take three of them. Sections 5 and 6 say `2RB3WR` and `3RB2WR` are hard to tell apart; this
section says `RB-RB-WR-WR-WR` and `WR-WR-RB-RB-RB` are not.

Caveat, and it is the same one as section 6: individually these permutations mostly don't clear
|t| > 2 either. The claim worth making is the grouped one — opens-with-RB against opens-with-WR —
which the second cell tests directly.

In [16]:
orderings = q("""
    SELECT league_key, strategy, points_vs_field, finish_rank, t_stat,
           seasons_positive, n_seasons
    FROM draft_strategy_summary
    WHERE strategy_kind = 'ordering' AND variant = 'actual' AND field_model = 'adp'
    ORDER BY league_key, points_vs_field DESC
""")
orderings["opens"] = orderings.strategy.str.split("-").str[0]
orderings["RBs"] = orderings.strategy.str.count("RB")

for league_key in ("sleeper", "espn"):
    table = orderings[orderings.league_key == league_key].drop(columns="league_key")
    print(f"\n=== {league_key} — forced opening sequences, best to worst ===")
    print(table.round(2).to_string(index=False))


=== sleeper — forced opening sequences, best to worst ===
      strategy  points_vs_field  finish_rank  t_stat  seasons_positive  n_seasons opens  RBs
RB-RB-WR-WR-WR            42.38         5.70    2.22                 8         11    RB    2
RB-RB-WR-WR-RB            30.44         5.89    1.88                 8         11    RB    3
RB-WR-WR-WR-RB            28.23         6.12    2.33                 9         11    RB    2
RB-RB-RB-WR-WR            28.00         5.91    1.32                 8         11    RB    3
RB-WR-RB-WR-WR            26.85         5.91    1.95                 8         11    RB    2
WR-RB-RB-WR-WR            25.62         5.96    1.03                 7         11    WR    2
WR-RB-WR-WR-RB            22.01         5.93    1.26                 6         11    WR    2
RB-WR-WR-RB-WR            17.84         6.17    1.57                 7         11    RB    2
RB-WR-RB-WR-RB            17.08         6.33    1.66                 8         11    RB    3
RB-RB-WR-RB

In [17]:
# The grouped claim: does opening with a back beat opening with a receiver?
rows = []
for league_key in ("sleeper", "espn"):
    table = orderings[orderings.league_key == league_key]
    rb_first = table[table.opens == "RB"].points_vs_field
    wr_first = table[table.opens == "WR"].points_vs_field
    three_rb = table[table.RBs == 3].points_vs_field
    two_rb = table[table.RBs == 2].points_vs_field
    rows.append({
        "league": league_key,
        "opens RB": rb_first.mean(),
        "opens WR": wr_first.mean(),
        "RB-first edge": rb_first.mean() - wr_first.mean(),
        "3 RB total": three_rb.mean(),
        "2 RB total": two_rb.mean(),
        "3-RB edge": three_rb.mean() - two_rb.mean(),
    })
pd.DataFrame(rows).set_index("league").round(1)

,opens RB,opens WR,RB-first edge,3 RB total,2 RB total,3-RB edge
league,,,,,,
sleeper,21.4,2.8,18.6,8.9,15.3,-6.5
espn,-5.4,-20.5,15.0,-12.6,-13.3,0.7


In [18]:
# "Every RB opener beats every WR-WR opener", checked. Also: how big is the ordering effect next to
# the composition effect it sits inside?
for league_key in ("sleeper", "espn"):
    table = orderings[orderings.league_key == league_key]
    rb_open = table[table.opens == "RB"].points_vs_field
    wr_wr_open = table[table.strategy.str.startswith("WR-WR")].points_vs_field
    holds = rb_open.min() > wr_wr_open.max()

    # "Sensible" = no all-in openings and no doubling up at QB or TE, i.e. the compositions a
    # human would actually be choosing between.
    all_compositions = ranking(league_key)
    sensible = all_compositions[
        ~all_compositions.strategy.str.contains(r"[45](?:RB|WR)|2QB|2TE")
    ].points_vs_field
    print(
        f"{league_key:>8}: worst RB-opener {rb_open.min():+.1f} vs best WR-WR opener "
        f"{wr_wr_open.max():+.1f} -> claim holds: {holds}\n"
        f"{'':>10}ordering spread {table.points_vs_field.max() - table.points_vs_field.min():.0f} pts; "
        f"spread across sensible compositions {sensible.max() - sensible.min():.0f} pts"
    )

 sleeper: worst RB-opener +1.5 vs best WR-WR opener +0.5 -> claim holds: True
          ordering spread 64 pts; spread across sensible compositions 61 pts


    espn: worst RB-opener -18.8 vs best WR-WR opener -24.3 -> claim holds: True
          ordering spread 44 pts; spread across sensible compositions 58 pts


In [19]:
orderings.groupby(["league_key", "opens"]).points_vs_field.agg(
    ["count", "mean", "min", "max"]
).round(1)

count  mean   min   max
league_key opens                         
espn       RB        10  -5.4 -18.8   7.1
           WR        10 -20.5 -36.5  -2.0
sleeper    RB        10  21.4   1.5  42.4
           WR        10   2.8 -21.6  25.6

<a id="field"></a>
## 8. Who else is at the table

Every number so far assumes the other eleven (or nine) managers draft straight off ADP and never
deviate. That is the cleanest way to ask "does departing from the market pay", and it is also the
friendliest, because a lone deviator competes with nobody for the position it is hoarding.

`field_model = 'mixed'` drops that assumption: every opponent gets its own randomly drawn opening
composition, three seeded replicates per draft. The strategies keep their relative order — the rank
correlation between the two fields is high — but the *level* moves, and it moves in a direction that
matters.

**Against a field that is itself deviating, every plan looks better than it did — and the
do-nothing control gains the most.** In the 12-team Sleeper league that is enough to move pure
best-available-by-ADP from a break-even control to the top of the board outright. In the 10-team
ESPN league it improves but stays mid-table, where a balanced opening still leads; the shallower
field there leaves less on the table for a disciplined drafter to collect.

The direction is the same in both, and it makes sense on reflection: when everyone else reaches to
fill a positional quota, they leave value on the board, and the drafter with no quota to fill is the
one who picks it up. It also matters more than the rest of this notebook, because a real draft room
is much closer to the mixed field than to the ADP one. Your leaguemates have watched the same videos.

In [20]:
agreement = []
for league_key in ("sleeper", "espn"):
    for variant in ("actual", "superflex"):
        both = ranking(league_key, variant, "adp").merge(
            ranking(league_key, variant, "mixed"), on="strategy", suffixes=(" adp", " mixed")
        )
        agreement.append({
            "league": league_key, "variant": variant, "strategies": len(both),
            "rank correlation": both["points_vs_field adp"].corr(
                both["points_vs_field mixed"], method="spearman"
            ),
        })
pd.DataFrame(agreement).set_index(["league", "variant"]).round(3)

strategies  rank correlation
league  variant                                
sleeper actual             37             0.916
        superflex          37             0.961
espn    actual             37             0.765
        superflex          37             0.821

In [21]:
for league_key in ("sleeper", "espn"):
    table = ranking(league_key, "actual", "mixed")
    keep = pd.concat([
        table.head(5), table[table.strategy.isin(["ADP", "3RB2WR"])], table.tail(3)
    ]).drop_duplicates("strategy").sort_values("points_vs_field", ascending=False)
    print(f"\n=== {league_key} — against a field that also deviates (of {len(table)}) ===")
    print(keep.round(2).to_string())


=== sleeper — against a field that also deviates (of 37) ===
        strategy  points_vs_field  finish_rank   win%  top3rd%  t_stat  seasons_positive  n_seasons
1            ADP            36.11         5.78  11.62    41.67    2.36                 9         11
2      2RB2WR1TE            34.90         5.88   8.84    42.42    3.00                 8         11
3      1QB2RB2WR            33.25         5.96   8.84    40.66    2.59                 9         11
4   1QB2RB1WR1TE            28.23         5.95   9.85    42.68    3.23                 8         11
5      1QB1RB3WR            27.42         6.02   8.84    41.16    1.44                 8         11
8         3RB2WR            24.13         6.10  10.10    35.10    1.80                 9         11
35     1QB2WR2TE           -33.62         7.21   6.57    23.48   -2.02                 3         11
36        3WR2TE           -35.88         7.05   7.07    26.52   -1.89                 2         11
37     2QB1WR2TE           -41.22     

In [22]:
# How far the do-nothing control moves once the rest of the room stops following ADP.
for league_key in ("sleeper", "espn"):
    moves = []
    for field_model in ("adp", "mixed"):
        table = ranking(league_key, "actual", field_model)
        moves.append((int(table.index[table.strategy == "ADP"][0]),
                      float(table.loc[table.strategy == "ADP", "points_vs_field"].iloc[0])))
    (adp_rank, adp_value), (mixed_rank, mixed_value) = moves
    print(f"{league_key:>8}: pure-ADP discipline ranks {adp_rank} vs a passive field "
          f"({adp_value:+.1f}) and {mixed_rank} vs a deviating one ({mixed_value:+.1f})")

 sleeper: pure-ADP discipline ranks 10 vs a passive field (+0.0) and 1 vs a deviating one (+36.1)
    espn: pure-ADP discipline ranks 16 vs a passive field (+0.0) and 16 vs a deviating one (+6.5)


<a id="superflex"></a>
## 9. The superflex counterfactual

Neither league is superflex (section 1). But the worry behind the question is a good one, and it is
worth knowing what the answer *would* be — not least so the answer is already in hand if either
league changes its settings.

`variant = 'superflex'` re-runs the entire sweep with one bench spot converted into a superflex slot,
so roster size and draft length are unchanged and the only thing that moves is quarterback
eligibility. The contrast is stark: the quarterback slope goes from statistically indistinguishable
from zero to the single largest and most significant effect anywhere in this notebook, positive in
almost every season, in both leagues.

So the instinct is right — it is just conditional on a setting these leagues don't have. **Under the
actual settings, a quarterback in the first five rounds is not something to reach for; under
superflex, two of them would be the highest-confidence finding here.**

Two caveats on the size of that number, both of which push the same way. The ADP board being drafted
from is a *1QB* board — FantasyPros and FFC price for the formats their users play — so the focal
team is buying quarterbacks at one-quarterback prices for a two-quarterback lineup, and real
superflex ADP has already repriced exactly that. Read the superflex figures as an **upper bound** on
what an early quarterback is worth, and the gap between the two variants as what the slot itself is
doing.

In [23]:
for position in ("QB", "RB"):
    rows = []
    for league_key in ("sleeper", "espn"):
        for variant in ("actual", "superflex"):
            frame = compositions[
                (compositions.league_key == league_key)
                & (compositions.variant == variant)
            ]
            rows.append({"league": league_key, "variant": variant, **trend(frame, position)})
    print(f"\n=== value of the Nth {position} in the opening, actual vs superflex ===")
    print(pd.DataFrame(rows).drop(columns="position")
          .set_index(["league", "variant"]).round(2).to_string())


=== value of the Nth QB in the opening, actual vs superflex ===
                   0 in opening  1 in opening  2 in opening  3 in opening  4 in opening  5 in opening  pts/extra pick     t     p
league  variant                                                                                                                  
sleeper actual            -5.20        -13.26        -31.56           NaN           NaN           NaN          -13.18 -1.72  0.12
        superflex        -35.06         28.12         59.73           NaN           NaN           NaN           47.40  5.44  0.00
espn    actual           -12.86          4.34         -7.65           NaN           NaN           NaN            2.61  0.34  0.74
        superflex        -27.05         18.60         30.60           NaN           NaN           NaN           28.83  5.99  0.00

=== value of the Nth RB in the opening, actual vs superflex ===
                   0 in opening  1 in opening  2 in opening  3 in opening  4 in opening  5

In [24]:
for league_key in ("sleeper", "espn"):
    table = ranking(league_key, "superflex", "mixed")
    keep = pd.concat([table.head(5), table[table.strategy == "ADP"]]).drop_duplicates("strategy")
    print(f"\n=== {league_key} / superflex — best openings against a deviating field ===")
    print(keep.round(2).to_string())


=== sleeper / superflex — best openings against a deviating field ===
        strategy  points_vs_field  finish_rank   win%  top3rd%  t_stat  seasons_positive  n_seasons
1      2QB2RB1WR            87.20         5.04  16.41    49.75    5.73                11         11
2      2QB1RB2WR            78.61         5.30  14.39    46.46    4.50                10         11
3         2QB3RB            73.64         5.33  15.40    50.25    3.69                10         11
4      2QB2RB1TE            69.62         5.32  12.37    47.73    4.52                10         11
5   2QB1RB1WR1TE            63.69         5.38  14.14    44.44    5.53                11         11
17           ADP            12.86         6.29  10.35    36.11    0.76                 6         11

=== espn / superflex — best openings against a deviating field ===
        strategy  points_vs_field  finish_rank   win%  top3rd%  t_stat  seasons_positive  n_seasons
1      2QB2RB1WR            68.68         4.48  17.88    55.1

<a id="doing"></a>
## 10. What to actually do

Pulling the sections together, with each claim carrying the section that supports it:

1. **Don't run the full house.** Three backs in five rounds ranks mid-table in Sleeper and near the
   bottom in ESPN (§5), and it spends its third premium pick in rounds 3-5, which is where running
   backs return least and hit least often (§3, §4).
2. **But do take a back early.** The scarcity the strategy is reaching for is real — it is just
   two rounds deep, not five (§2). Every forced sequence that opens RB beat every sequence that
   opened WR-WR (§7).
3. **One or two backs, then receivers.** Both leagues' best openings sit at 1-2 RB; zero is bad and
   four or five is worse (§6). Sleeper's half-PPR, two-flex setup tolerates the extra back better
   than ESPN's full-PPR single-flex does (§2, §3). If you want one opening to hold in both leagues,
   `2RB2WR1TE` is the only one that clears significance in each of them independently (§6).
4. **Don't reach for a quarterback, and don't worry about superflex.** Neither league has the slot
   (§1). If one ever adds it, take two quarterbacks early and stop reading anything else here (§9).
5. **The least glamorous finding travels furthest:** once the rest of the room is also running
   strategies, discipline gets cheaper to hold and quotas get more expensive. In Sleeper that makes
   plain best-available-by-ADP the top plan outright; in ESPN it stays mid-table behind a balanced
   opening (§8). Either way — deviate to break a tie, not on principle.

The cell below assembles the per-league recommendation from the tables rather than from that list,
so if a rebuild moves the numbers it moves here too.

In [25]:
for league_key in ("sleeper", "espn"):
    settings = q("SELECT * FROM league_settings WHERE league_key = ?", [league_key]).iloc[0]
    passive = ranking(league_key, "actual", "adp")
    deviating = ranking(league_key, "actual", "mixed")
    best = passive.iloc[0]
    rb_first = orderings[(orderings.league_key == league_key) & (orderings.opens == "RB")]
    wr_first = orderings[(orderings.league_key == league_key) & (orderings.opens == "WR")]

    print(f"\n{'=' * 68}\n{league_key.upper()} — {settings.team_count} teams, "
          f"{settings.rec_pts} PPR, {settings.flex_slots} flex, "
          f"{settings.superflex_slots} superflex\n{'=' * 68}")
    print(f"  best opening vs a passive field : {best.strategy} "
          f"({best.points_vs_field:+.0f} pts, t={best.t_stat:.2f})")
    print(f"  best opening vs a deviating field: {deviating.iloc[0].strategy} "
          f"({deviating.iloc[0].points_vs_field:+.0f} pts, t={deviating.iloc[0].t_stat:.2f})")
    print(f"  full house (3RB2WR)             : rank {int(passive.index[passive.strategy == '3RB2WR'][0])}"
          f" of {len(passive)}")
    print(f"  open RB vs open WR              : "
          f"{rb_first.points_vs_field.mean() - wr_first.points_vs_field.mean():+.0f} pts for RB first")
    print(f"  QB in the first five rounds     : "
          f"{'yes — superflex' if settings.superflex_slots else 'no — one QB slot, no flex eligibility'}")


SLEEPER — 12 teams, 0.5 PPR, 2 flex, 0 superflex
  best opening vs a passive field : 2RB3WR (+28 pts, t=2.58)
  best opening vs a deviating field: ADP (+36 pts, t=2.36)
  full house (3RB2WR)             : rank 12 of 37
  open RB vs open WR              : +19 pts for RB first
  QB in the first five rounds     : no — one QB slot, no flex eligibility



ESPN — 10 teams, 1.0 PPR, 1 flex, 0 superflex
  best opening vs a passive field : 1QB1RB2WR1TE (+41 pts, t=2.10)
  best opening vs a deviating field: 1QB2RB2WR (+34 pts, t=3.31)
  full house (3RB2WR)             : rank 27 of 37
  open RB vs open WR              : +15 pts for RB first
  QB in the first five rounds     : no — one QB slot, no flex eligibility


### The 2026 board

Strategy decides *which position* to take; `draft_value` decides which player, by pricing this
year's projection against what that ADP slot has historically returned. `projected_surplus_rank` is
the draft-board column — at any given pick, who is expected to return the most over what he costs.

Shown per position so it can be read the way a draft actually goes: you are picking within a
position tier, not off one global list.

In [26]:
live_season = q("SELECT MAX(season) AS s FROM draft_value").s.iloc[0]
board = q("""
    SELECT position, player_name, ROUND(consensus_adp, 1) AS adp,
           ROUND(projected_surplus, 1) AS surplus, CAST(projected_surplus_rank AS INT) AS rank
    FROM draft_value
    WHERE league_key = 'sleeper' AND season = ? AND projected_surplus IS NOT NULL
      AND consensus_adp <= 120
    QUALIFY ROW_NUMBER() OVER (PARTITION BY position ORDER BY projected_surplus DESC) <= 8
    ORDER BY position, projected_surplus DESC
""", [int(live_season)])
print(f"{live_season} — best value per position inside the first 10 rounds (Sleeper scoring)")
board

2026 — best value per position inside the first 10 rounds (Sleeper scoring)


,position,player_name,adp,surplus,rank
0,QB,Jalen Hurts,70.0,51.6,2
1,QB,Josh Allen,25.0,25.2,12
2,QB,Bo Nix,113.0,10.5,27
3,QB,Caleb Williams,86.8,7.3,32
4,QB,Jaxson Dart,98.6,3.1,35
5,QB,Justin Herbert,91.5,-5.8,140
6,QB,Drake Maye,50.9,-7.3,156
7,QB,Jayden Daniels,65.4,-7.4,166
8,RB,Jeremiyah Love,27.7,59.4,1
9,RB,D'Andre Swift,48.7,36.6,4


---

## Adding to this notebook

The simulator lives in `src/gold/draft_strategy.py`, not here — its docstring carries the full
method and the caveats, and `python -m src.gold.draft_strategy` reprints the summary tables. This
notebook only reads `draft_strategy_results` / `draft_strategy_summary` and the models they were
built from.

`q()` opens a read-only connection, runs, and closes it — so nothing here can hold a lock that
blocks `scripts/build_warehouse.sh`, and nothing here can write to the warehouse. See
`notebooks/README.md` for the conventions.

Useful while exploring:

```python
tables("gold")
columns("draft_strategy_summary")
peek("draft_strategy_results")
```

Worth testing if this gets picked up again:

- **Keeper/dynasty rules.** Everything here is a redraft simulation.
- **In-season roster churn.** The simulation drafts and then freezes; it has no waiver wire, and a
  strategy that leaves you thin at a position is punished less here than in a real season.
- **Weekly lineups rather than season totals.** Teams are scored on the hindsight-best lineup, which
  is the same fiction for every strategy but flatters high-variance rosters slightly.